# WarpTorch: Alcubierre Warp Drive End-to-End Simulation
This notebook demonstrates how to initialize an Alcubierre warp metric, compute its stress-energy tensor, calculate kinematic scalars, and evaluate energy conditions using PyTorch.

In [ ]:
import sys
import os

# Add parent directory to path to import core modules
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
import torch
import plotly.graph_objects as go
import numpy as np

from core.metrics.alcubierre import get_alcubierre_metric
from core.solver.energy import get_energy_tensor
from core.analyzer.scalars import get_kinematic_scalars
from core.analyzer.energy_conditions import evaluate_energy_conditions
from core.visualizer.slicing import get_2d_slice
from core.utils import get_best_device

device = get_best_device()
print(f"Using device: {device}")

## 1. Spacetime Grid & Metric Initialization
We define a 4D grid `[T, X, Y, Z]` and set up an Alcubierre warp bubble traveling along the X-axis at $v = 1.5c$ (superluminal warp drive).

In [ ]:
grid_size = (1, 64, 64, 64)
grid_scale = (0.1, 0.5, 0.5, 0.5)
world_center = (0.0, 16.0, 16.0, 16.0)

print("Generating Alcubierre metric...")
metric = get_alcubierre_metric(
    grid_size=grid_size,
    world_center=world_center,
    v=1.5,
    R=6.0,
    sigma=4.0,
    grid_scale=grid_scale,
    device=device
)
print(f"Metric tensor generated successfully on {metric.device}.")

## 2. Einstein's Field Equations Solver
We compute the contravariant Stress-Energy Tensor $T^{\mu\nu}$ using vectorized 4th-order finite difference stencils.

In [ ]:
print("Evaluating stress-energy tensor pipeline...")
energy_tensor = get_energy_tensor(metric)
print("Stress-Energy calculation complete.")

## 3. Kinematic Expansion & Energy Conditions
Extract the expansion scalar ($\theta$) and run the Null Energy Condition (NEC) validator to map negative energy density distributions.

In [ ]:
scalars = get_kinematic_scalars(metric)
expansion = scalars["expansion"]

print("Evaluating Null Energy Condition (NEC)...")
nec_results = evaluate_energy_conditions(energy_tensor, metric, condition="Null", num_angular=100)
print(f"Is NEC violated? {nec_results['is_violated']}")

## 4. Interactive Plotly Visualizations
Rendering a 2D mid-plane slice ($XY$-plane) for both Energy Density ($T^{00}$) and Spacetime Expansion ($\theta$).

In [ ]:
t00_slice = get_2d_slice(energy_tensor, component=(0, 0), slice_plane='xy')
expansion_slice = expansion[0, :, :, grid_size[3] // 2].detach().cpu().numpy()

x_coords = (np.arange(grid_size[1]) * grid_scale[1]) - world_center[1]
y_coords = (np.arange(grid_size[2]) * grid_scale[2]) - world_center[2]

fig_t00 = go.Figure(data=go.Heatmap(
    z=t00_slice.T, x=x_coords, y=y_coords,
    colorscale='RdBu', zmid=0,
    colorbar=dict(title="Energy Density (J/m³)")
))
fig_t00.update_layout(
    title="Alcubierre Warp Bubble: Energy Density ($T^{00}$) Slice",
    xaxis_title="X (Meters)", yaxis_title="Y (Meters)"
)
fig_t00.show()

fig_exp = go.Figure(data=go.Heatmap(
    z=expansion_slice.T, x=x_coords, y=y_coords,
    colorscale='Viridis',
    colorbar=dict(title="Expansion Scalar θ")
))
fig_exp.update_layout(
    title="Spacetime Expansion & Contraction Profile (θ)",
    xaxis_title="X (Meters)", yaxis_title="Y (Meters)"
)
fig_exp.show()